In [13]:
from agent.preferences import (
    init_preference_state,
    update_preferences_from_input,
    check_missing_required,
    generate_followup_question,
)
from agent.retrieval import load_listings, filter_listings
from agent.ranking import rank_listings
from agent.explain import explain_listing


def main():
    df = load_listings("listings.csv")

    state = init_preference_state()

    # Temporary structured input for testing
    user_input = {
        "hard_constraints": {
            "max_price": 250,
            "location": "Brooklyn",
            "guests": 2,
            "room_type": "Entire home/apt",
        },
        "soft_preferences": {
            "amenities": ["wifi", "kitchen"],
            "vibe": "quiet",
            "purpose": "remote work",
            "preferred_area": "Brooklyn",
        }
    }

    state = update_preferences_from_input(state, user_input)

    missing = check_missing_required(state)
    if missing:
        print(generate_followup_question(missing))
        return

    filtered = filter_listings(df, state)

    if filtered.empty:
        print("No exact matches found. Next step: relax soft constraints or expand search.")
        return

    ranked = rank_listings(filtered, state, top_k=5)

    print("\nTop recommendations:\n")
    for _, row in ranked.iterrows():
        print(f"Listing: {row.get('name', 'Unknown')}")
        print(f"Location: {row.get('location', 'Unknown')}")
        print(f"Price: ${row.get('price', 'N/A')}")
        print(f"Score: {row.get('score', 0):.2f}")
        print(explain_listing(row, state))
        print("-" * 50)


if __name__ == "__main__":
    main()

No exact matches found. Next step: relax soft constraints or expand search.
